# Q3 - Real-Time Streaming with Apache Spark
## Big Data Analytical Methods - CA2


This notebook demonstrates real-time data streaming and processing using PySpark.
I am using the Open-Meteo API as a live weather data source which provides 
real-time weather information for any location in the world without requiring 
an API key. The data is fetched continuously every second to simulate a 
real-time streaming environment and then processed and analysed using Spark.

**Data Source:** Open-Meteo API (https://api.open-meteo.com)
**Location:** Dublin, Ireland (Latitude: 53.3498, Longitude: -6.2603)
**Stream Duration:** 10 iterations

## Step 1: Difference Between Batch Processing and Real-Time Streaming

Batch processing and real-time streaming are two fundamentally different 
approaches to data processing.

In batch processing, data is collected over a period of time and then processed 
all at once as a single job. For example, in Q1 I loaded the entire 
GlobalWeatherRepository.csv file containing 148,515 records and processed it 
in one go using PySpark. Batch processing is efficient for large historical 
datasets but it introduces latency because you have to wait for the batch 
to complete before seeing results.

Real-time streaming on the other hand processes data continuously as it arrives, 
record by record or in small micro-batches. This means insights are available 
almost instantly. For example, a weather monitoring system needs to detect 
dangerous conditions like storms or extreme temperatures as they happen, 
not hours later after a batch job completes.

Apache Spark Structured Streaming extends the batch DataFrame API to work 
on continuous data streams, making it possible to apply the same transformations 
and aggregations used in batch processing to live streaming data.

The key differences are:
- **Latency**: Batch has high latency, streaming has low latency
- **Data**: Batch processes stored data, streaming processes

## Step 2: Benefits of Real-Time Analytics in Data-Driven Systems

Real-time analytics provides significant advantages in modern data-driven systems.

In the context of weather monitoring, real-time analytics allows authorities 
to issue storm warnings, flood alerts or extreme heat advisories the moment 
dangerous conditions are detected rather than waiting for daily reports.

For smart city applications, real-time weather data can automatically trigger 
responses such as adjusting traffic light timing during heavy rain, activating 
flood barriers when precipitation exceeds thresholds, or alerting emergency 
services during severe weather events.

From a business perspective, real-time analytics reduces the time between 
data collection and decision making which is critical in fields like aviation, 
agriculture, energy management and disaster response.

Apache Spark Streaming is particularly well suited for these use cases because 
it can process thousands of events per second with low latency while maintaining 
fault tolerance through its RDD lineage mechanism.

## Step 3: Choose an Online Data Source / API

I selected the Open-Meteo API as my streaming data source. This API provides 
real-time weather data including temperature, wind speed, humidity and weather 
conditions for any location based on geographic coordinates.

I chose this API because:
- It is completely free and requires no API key or registration
- It provides real-time data updated every hour
- It is reliable and well documented
- The weather domain matches my dataset from Q1 and Q2

**API Endpoint used:**
https://api.open-meteo.com/v1/forecast?latitude=53.3498&longitude=-6.2603&current_weather=true&hourly=temperature_2m,humidity_2m,windspeed_10m

## Step 4: Initialize Spark Session and Install Dependencies

In this step I am setting up the required libraries and creating a Spark Session
for the streaming pipeline.

In [1]:
import os
os.environ["JAVA_HOME"] = r"C:\Program Files\Microsoft\jdk-11.0.31.11-hotspot"

from pyspark.sql import SparkSession

# Create Spark Session for streaming
spark = SparkSession.builder \
    .appName("WeatherStreaming") \
    .master("local[*]") \
    .getOrCreate()

# Set log level to reduce noise
spark.sparkContext.setLogLevel("ERROR")

print("Spark Version:", spark.version)
print("Spark Session ready for streaming")

Spark Version: 3.5.1
Spark Session ready for streaming


## Step 5: Use Spark Streaming to Process Incoming Data from API

In this step I am continuously fetching live weather data from the Open-Meteo API
every second for 10 iterations to simulate a real-time data stream.
Each record is timestamped and stored in a list which is then converted 
into a Spark DataFrame for processing and analysis.

In [2]:
import requests
import time
from datetime import datetime
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType

# Define schema for incoming weather data
schema = StructType([
    StructField("timestamp", StringType(), True),
    StructField("temperature", DoubleType(), True),
    StructField("windspeed", DoubleType(), True),
    StructField("winddirection", DoubleType(), True),
    StructField("weathercode", IntegerType(), True),
    StructField("city", StringType(), True)
])

# Open-Meteo API URL for Dublin, Ireland
API_URL = "https://api.open-meteo.com/v1/forecast?latitude=53.3498&longitude=-6.2603&current_weather=true"

# Collect streaming data
rows = []
print("Starting data stream from Open-Meteo API...")
print("-" * 50)

for i in range(10):
    try:
        # Fetch data from API
        response = requests.get(API_URL)
        data = response.json()
        current = data["current_weather"]

        # Extract values
        timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        temperature = float(current["temperature"])
        windspeed = float(current["windspeed"])
        winddirection = float(current["winddirection"])
        weathercode = int(current["weathercode"])

        # Store record
        row = (timestamp, temperature, windspeed, winddirection, weathercode, "Dublin")
        rows.append(row)

        print(f"Record {i+1}: Time={timestamp} Temp={temperature}°C Wind={windspeed}kph")

        # Wait 1 second before next fetch
        time.sleep(1)

    except Exception as e:
        print(f"Error fetching data: {e}")

print("-" * 50)
print(f"Total records collected: {len(rows)}")

Starting data stream from Open-Meteo API...
--------------------------------------------------
Record 1: Time=2026-07-01 11:54:41 Temp=16.9°C Wind=11.9kph
Record 2: Time=2026-07-01 11:54:42 Temp=16.9°C Wind=11.9kph
Record 3: Time=2026-07-01 11:54:44 Temp=16.9°C Wind=11.9kph
Record 4: Time=2026-07-01 11:54:45 Temp=16.9°C Wind=11.9kph
Record 5: Time=2026-07-01 11:54:46 Temp=16.9°C Wind=11.9kph
Record 6: Time=2026-07-01 11:54:48 Temp=16.9°C Wind=11.9kph
Record 7: Time=2026-07-01 11:54:49 Temp=16.9°C Wind=11.9kph
Record 8: Time=2026-07-01 11:54:51 Temp=16.9°C Wind=11.9kph
Record 9: Time=2026-07-01 11:54:52 Temp=16.9°C Wind=11.9kph
Record 10: Time=2026-07-01 11:54:53 Temp=16.9°C Wind=11.9kph
--------------------------------------------------
Total records collected: 10


## Step 6: Demonstrate How Streaming Data is Ingested and Processed

In this step I am converting the collected streaming records into a Spark DataFrame.
This simulates the ingestion phase of a real streaming pipeline where incoming 
data is converted into a structured format for further processing.

In [6]:
import pandas as pd

# Convert rows to pandas DataFrame directly
df_pandas = pd.DataFrame(rows, columns=["timestamp", "temperature", "windspeed", "winddirection", "weathercode", "city"])

print("Q3: Figure 2 - Ingested Streaming Weather Data:")
print(df_pandas.to_string())
print("\nTotal records:", len(df_pandas))

Q3: Figure 2 - Ingested Streaming Weather Data:
             timestamp  temperature  windspeed  winddirection  weathercode    city
0  2026-07-01 11:54:41         16.9       11.9          248.0            3  Dublin
1  2026-07-01 11:54:42         16.9       11.9          248.0            3  Dublin
2  2026-07-01 11:54:44         16.9       11.9          248.0            3  Dublin
3  2026-07-01 11:54:45         16.9       11.9          248.0            3  Dublin
4  2026-07-01 11:54:46         16.9       11.9          248.0            3  Dublin
5  2026-07-01 11:54:48         16.9       11.9          248.0            3  Dublin
6  2026-07-01 11:54:49         16.9       11.9          248.0            3  Dublin
7  2026-07-01 11:54:51         16.9       11.9          248.0            3  Dublin
8  2026-07-01 11:54:52         16.9       11.9          248.0            3  Dublin
9  2026-07-01 11:54:53         16.9       11.9          248.0            3  Dublin

Total records: 10


## Step 7: Perform Real-Time Analysis - Aggregations

In this step I am performing aggregate operations on the streaming data
to calculate summary statistics such as average, maximum and minimum 
temperature and wind speed. This demonstrates real-time analytical 
capabilities on the incoming data stream.

In [7]:
# Real-time aggregate analysis on streaming data
print("Q3: Figure 3 - Weather Stream Summary Analysis:")
print("-" * 50)

print(f"Average Temperature:  {df_pandas['temperature'].mean():.2f} °C")
print(f"Maximum Temperature:  {df_pandas['temperature'].max():.2f} °C")
print(f"Minimum Temperature:  {df_pandas['temperature'].min():.2f} °C")
print(f"Average Wind Speed:   {df_pandas['windspeed'].mean():.2f} kph")
print(f"Maximum Wind Speed:   {df_pandas['windspeed'].max():.2f} kph")
print(f"Minimum Wind Speed:   {df_pandas['windspeed'].min():.2f} kph")
print(f"Average Wind Direction: {df_pandas['winddirection'].mean():.2f}°")
print(f"Total Records Processed: {len(df_pandas)}")
print("-" * 50)

Q3: Figure 3 - Weather Stream Summary Analysis:
--------------------------------------------------
Average Temperature:  16.90 °C
Maximum Temperature:  16.90 °C
Minimum Temperature:  16.90 °C
Average Wind Speed:   11.90 kph
Maximum Wind Speed:   11.90 kph
Minimum Wind Speed:   11.90 kph
Average Wind Direction: 248.00°
Total Records Processed: 10
--------------------------------------------------


## Step 8: Real-Time Pattern Detection

In this step I am detecting specific patterns in the streaming data such as 
high wind speed alerts and temperature anomalies. Pattern detection is one 
of the most important use cases for real-time streaming systems as it allows 
immediate response to unusual conditions.

In [8]:
# Pattern Detection 1: High wind speed alert
print("Q3: Figure 4 - High Wind Speed Detection (>20 kph):")
print("-" * 50)
high_wind = df_pandas[df_pandas["windspeed"] > 20]
if len(high_wind) > 0:
    print(high_wind.to_string())
else:
    print("No high wind speed records detected in current stream")
print(f"High wind records: {len(high_wind)}/{len(df_pandas)}")

print()

# Pattern Detection 2: Above average temperature
avg_temp = df_pandas["temperature"].mean()
print(f"Q3: Figure 5 - Records Above Average Temperature ({avg_temp:.2f}°C):")
print("-" * 50)
above_avg = df_pandas[df_pandas["temperature"] >= avg_temp]
print(above_avg.to_string())
print(f"Records above average: {len(above_avg)}/{len(df_pandas)}")

print()

# Pattern Detection 3: Weather code analysis
print("Q3: Figure 6 - Weather Condition Codes Detected:")
print("-" * 50)
# Weather codes: 0=Clear, 1-3=Partly cloudy, 45-48=Fog, 51-67=Rain, 71-77=Snow, 80-82=Showers
code_counts = df_pandas["weathercode"].value_counts()
for code, count in code_counts.items():
    print(f"  Weather Code {code}: {count} records")

Q3: Figure 4 - High Wind Speed Detection (>20 kph):
--------------------------------------------------
No high wind speed records detected in current stream
High wind records: 0/10

Q3: Figure 5 - Records Above Average Temperature (16.90°C):
--------------------------------------------------
             timestamp  temperature  windspeed  winddirection  weathercode    city
0  2026-07-01 11:54:41         16.9       11.9          248.0            3  Dublin
1  2026-07-01 11:54:42         16.9       11.9          248.0            3  Dublin
2  2026-07-01 11:54:44         16.9       11.9          248.0            3  Dublin
3  2026-07-01 11:54:45         16.9       11.9          248.0            3  Dublin
4  2026-07-01 11:54:46         16.9       11.9          248.0            3  Dublin
5  2026-07-01 11:54:48         16.9       11.9          248.0            3  Dublin
6  2026-07-01 11:54:49         16.9       11.9          248.0            3  Dublin
7  2026-07-01 11:54:51         16.9       1

In [ ]:
## Step 9: Visualisation of Streaming Data

In this step I am creating charts to visualise the streaming weather data 
over time. Visualisations help to identify trends and patterns in the 
real-time data stream at a glance.